## 1. ACCESSING DATA

In [1]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt


pd.set_option('display.max_columns', None)


In [4]:
df = pd.read_csv('../files/hr_raw_data_v0.csv', index_col=0)

In [5]:
df.head(1)

,age,attrition,businesstravel,dailyrate,department,distancefromhome,education,educationfield,employeecount,employeenumber,environmentsatisfaction,gender,hourlyrate,jobinvolvement,joblevel,jobrole,jobsatisfaction,maritalstatus,monthlyincome,monthlyrate,numcompaniesworked,over18,overtime,percentsalaryhike,performancerating,relationshipsatisfaction,standardhours,stockoptionlevel,totalworkingyears,trainingtimeslastyear,worklifebalance,yearsatcompany,yearsincurrentrole,yearssincelastpromotion,yearswithcurrmanager,sameasmonthlyincome,datebirth,salary,roledepartament,numberchildren,remotework
0,51,No,NaN,2015.722222,NaN,6,3,NaN,1,1,1,0,NaN,3,5,resEArch DIREcToR,3,NaN,"16280,83$","42330,17$",7,Y,No,13,"3,0",3,Full Time,0,NaN,5,"3,0",20,NaN,15,15,"16280,83$",1972,"195370,00$",NaN,NaN,Yes


## 2. STRUCTURAL DEFINITION AND STRUCTURAL CLEANING

### 2.1 STRUCTURAL DEFINITION

In [6]:
#COLUMN RENAMES:

title_mapping = {"employeenumber": "employee_number",
                "gender": "gender",
                "datebirth": "birth_year",
                "age": "age",
                "maritalstatus": "marital_status",
                "jobrole": "job_title",
                "department": "department",
                "attrition": "departured",
                "standardhours": "standard_hours",
                "monthlyincome": "monthly_income",
                "remotework": "remote",
                "businesstravel": "business_travel",
                "dailyrate": "daily_rate",
                "distancefromhome": "dist_home",
                "educationfield": "education_field",
                "education": "education_scale",
                "environmentsatisfaction": "env_sat_rate",
                "jobinvolvement": "job_involvement",
                "joblevel": "job_level",
                "jobsatisfaction": "job_sat_rate",
                "numcompaniesworked": "num_comp_worked",
                "overtime": "over_time",
                "percentsalaryhike": "perc_salary_hike",
                "performancerating": "perf_rate",
                "relationshipsatisfaction": "relationship_sat_rate",
                "stockoptionlevel": "stock_opt_level",
                "totalworkingyears": "tot_working_year",
                "trainingtimeslastyear": "traning_times_last_year",
                "worklifebalance": "work_life_balance",
                "yearsatcompany": "year_at_comp",
                "yearssincelastpromotion": "year_last_promotion",
                "yearswithcurrmanager": "year_current_mngr",
                "salary": "annual_salary"}

In [7]:
#CATEGORIES: 

columns_personal =      ['employee_number', 
                        'gender', 
                        'birth_year', 
                        'age', 
                        'marital_status',
                        'dist_home']

columns_job =           ['job_title',
                        'department',
                        'departured',
                        'year_at_comp',
                        'standard_hours',
                        'remote',
                        'business_travel',
                        'over_time', 
                        'job_level', 
                        'stock_opt_level', 
                        'traning_times_last_year', 
                        'perf_rate',
                        'year_last_promotion',
                        'year_current_mngr']

columns_education =     ['education_field',
                        'education_scale']


columns_income =        ['annual_salary',
                        'monthly_income',
                        'daily_rate',
                        'perc_salary_hike']

columns_satisfaction =  ['env_sat_rate',
                        'job_involvement',
                        'job_sat_rate',
                        'relationship_sat_rate',
                        'work_life_balance']

columns_emp_bgd =       ['num_comp_worked',
                        'tot_working_year']

In [8]:
#COLUMNS TO DROP 

drop_colums = ['yearsincurrentrole', 'roledepartament', 'sameasmonthlyincome','numberchildren', 'hourlyrate', 'monthlyrate']

### 2.2 STRUCTURAL CLEANING

- Renaming columns as per dictionary
- Reordering columns as per new names

In [9]:
df = df.drop(columns=drop_colums)

In [10]:
# COLUMN RENAME
df = df.rename(columns={k: v for k, v in title_mapping.items() if k in df.columns}) 


In [11]:
# COLUMN REORDER

new_order_columns = columns_personal+columns_job+columns_education+columns_income+columns_satisfaction+columns_emp_bgd

def reorder_columns(df, list_columns):

    #Reorder columns as per provided list, add all the missing ones at the end.
    
    try:
        df = df[new_order_columns]
        extra_columns = [col for col in df.columns if col not in new_order_columns]
        df = df[ new_order_columns + extra_columns ]
    
    except KeyError as e:
        print(f"KeyError: {e}")
        missing_columns = [col for col in new_order_columns if col not in df.columns]
        print(f"Missing columns: {missing_columns}")

        extra_columns = [col for col in df.columns if col not in new_order_columns]
        df = df[ new_order_columns + extra_columns ]
    
    return df

# CALLING THE REORDER FUNCCION

df = reorder_columns(df, new_order_columns)

In [12]:
#compruebo duplicados antes:
df.duplicated(keep=False).sum()

np.int64(128)

In [13]:
df.shape

(1678, 33)

In [14]:
#Consultamos los duplicados, nos muestra todas las filas duplicadas salvo la primera aparicion. 
# Tenemos 64 filas duplicadas

df[df.duplicated(subset='employee_number', keep='first')].sort_values('employee_number')

,employee_number,gender,birth_year,age,marital_status,dist_home,job_title,department,departured,year_at_comp,standard_hours,remote,business_travel,over_time,job_level,stock_opt_level,traning_times_last_year,perf_rate,year_last_promotion,year_current_mngr,education_field,education_scale,annual_salary,monthly_income,daily_rate,perc_salary_hike,env_sat_rate,job_involvement,job_sat_rate,relationship_sat_rate,work_life_balance,num_comp_worked,tot_working_year
1656,9,1,1982,41,Married,2,mANAGEr,NaN,No,18,Full Time,True,NaN,No,4,1,2,"3,0",11,8,NaN,5,"165950,00$","13829,17$",1712.182540,16,2,3,1,2,"3,0",7,"22,0"
1652,61,0,1987,36,Single,5,lAboratORy TeChNiCiaN,NaN,No,13,Full Time,1,NaN,No,2,0,3,"3,0",3,7,NaN,2,"59140,00$","4928,33$",610.174603,16,4,3,2,4,"4,0",8,"16,0"
1676,76,1,1976,47,Divorced,4,maNufACTURING DIREctOr,NaN,No,22,Part Time,Yes,travel_rarely,Yes,3,1,4,NaN,14,10,Life Sciences,3,"100071,84$","8339,32$",1032.487286,12,3,2,2,3,"3,0",8,NaN
1649,108,1,1994,29,Divorced,21,maNufaCturing direcTOr,NaN,No,10,NaN,0,travel_rarely,No,3,1,1,"3,0",8,8,Life Sciences,4,NaN,"8339,32$",1032.487286,11,2,4,1,3,"3,0",1,"10,0"
1616,112,1,1993,30,NaN,5,SalES ExeCuTIVe,NaN,No,10,Part Time,True,travel_rarely,No,3,1,2,"3,0",7,4,NaN,3,"100071,84$","8339,32$",1032.487286,12,2,3,4,3,"3,0",2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1647,1532,0,1989,34,NaN,-37,SalES exEcutIve,NaN,Yes,5,NaN,1,non-travel,Yes,2,0,3,"3,0",0,4,Marketing,3,NaN,"4420,00$",547.238095,13,1,4,4,2,"2,0",8,NaN
1637,1567,0,1988,35,NaN,16,HEalTHCArE rEpresEnTaTIvE,NaN,No,8,Part Time,1,travel_rarely,Yes,3,0,2,"3,0",0,0,Life Sciences,3,"100071,84$",NaN,1032.487286,12,4,3,3,3,"3,0",4,"10,0"
1638,1568,1,1975,48,Married,2,sALES EXEcuTIVe,NaN,No,9,Part Time,True,travel_rarely,No,2,1,2,"3,0",6,7,NaN,5,"40510,00$","3375,83$",417.960317,14,2,3,4,1,"3,0",2,"14,0"
1657,1569,0,1978,45,NaN,2,sAles executiVe,NaN,No,8,Part Time,False,travel_rarely,No,2,1,3,"3,0",3,7,Other,3,"48050,00$",NaN,495.753968,19,4,3,2,2,"4,0",0,NaN


In [15]:
df.drop_duplicates(keep = 'first', inplace= True)

In [16]:
#compruebo duplicados despues:
df.duplicated(keep=False).sum()

np.int64(0)

In [17]:
df.shape

(1614, 33)

In [18]:
df.head(1)


,employee_number,gender,birth_year,age,marital_status,dist_home,job_title,department,departured,year_at_comp,standard_hours,remote,business_travel,over_time,job_level,stock_opt_level,traning_times_last_year,perf_rate,year_last_promotion,year_current_mngr,education_field,education_scale,annual_salary,monthly_income,daily_rate,perc_salary_hike,env_sat_rate,job_involvement,job_sat_rate,relationship_sat_rate,work_life_balance,num_comp_worked,tot_working_year
0,1,0,1972,51,NaN,6,resEArch DIREcToR,NaN,No,20,Full Time,Yes,NaN,No,5,0,5,"3,0",15,15,NaN,3,"195370,00$","16280,83$",2015.722222,13,1,3,3,3,"3,0",7,NaN


## 3. NULLS MANAGEMENT

In [19]:
# NULLS BY COLUMN (%)
df.isnull().sum() / df.shape[0] * 100

employee_number             0.000000
gender                      0.000000
birth_year                  0.000000
age                         0.000000
marital_status             40.334572
dist_home                   0.000000
job_title                   0.000000
department                 81.288724
departured                  0.000000
year_at_comp                0.000000
standard_hours             20.941760
remote                      0.000000
business_travel            47.831475
over_time                  41.883519
job_level                   0.000000
stock_opt_level             0.000000
traning_times_last_year     0.000000
perf_rate                  12.081784
year_last_promotion         0.000000
year_current_mngr           0.000000
education_field            46.158612
education_scale             0.000000
annual_salary              16.976456
monthly_income             28.996283
daily_rate                  0.000000
perc_salary_hike            0.000000
env_sat_rate                0.000000
j

## 4. EXPORTING NEW DATA

In [20]:
df.to_csv("../files/hr_raw_data_v1.csv", index=False)
